# cough-vision -- Pulmonary TB Detection from Chest X-rays

> **Kaggle T4x2 DDP Training Notebook** | DenseNet-121 + Conv-stem ViT-S/16 Ensemble
> WHO TPP target: >=90% sensitivity at >=70% specificity

## Multi-GPU strategy: DistributedDataParallel (DDP)
```
mp.spawn -> 2x ddp_worker (one per T4 GPU, NCCL backend)
  DistributedWeightedSampler  -- unique balanced shard per GPU
  torch.cuda.amp.GradScaler   -- fp16 on each GPU
  dist.all_gather_object      -- AUC from ALL validation samples
  dist.broadcast              -- early-stop signal synced
  rank-0 only                 -- checkpoint save + W&B log
```

## Notebook flow
0. Environment + GPU detection
1. Install dependencies
2. Clone source from GitHub (tripathiji1312/cough-vision)
3. Dataset paths + CSV builder
4. Preprocessing sanity check
5. Hyperparameters
6. DDP utilities
7. Model smoke-test
8. DDP training worker
9. Launch training
10. Training curves
11. Evaluation -- WHO TPP report
12. Grad-CAM visualisation
13. Per-site calibration
14. ONNX export
15. Save artefacts


---
## 0 . Environment -- GPU detection

In [ ]:
import os, sys, subprocess, pathlib, platform, socket, datetime

IS_KAGGLE = os.path.exists('/kaggle/input')
WORK_DIR  = pathlib.Path('/kaggle/working') if IS_KAGGLE else pathlib.Path('.')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f'Platform : {platform.platform()}')
print(f'Python   : {sys.version.split()[0]}')
print(f'Kaggle   : {IS_KAGGLE}')

import torch

WORLD_SIZE = torch.cuda.device_count()
DEVICE     = 'cuda' if WORLD_SIZE > 0 else 'cpu'

print(f'\nGPU count : {WORLD_SIZE}')
for i in range(WORLD_SIZE):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}  : {p.name}  VRAM={p.total_memory/1e9:.1f} GB')

print(f'\nPyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.version.cuda}')

if WORLD_SIZE > 1:
    print(f'\nT4x2 detected -- DDP (NCCL) will be used.')
    print('NCCL_P2P_DISABLE=1 set for Kaggle PCIe topology')
elif WORLD_SIZE == 1:
    print('\nSingle GPU -- calling ddp_worker directly')
else:
    print('\nCPU mode -- set GPU T4x2 in notebook settings!')

import random, numpy as np
MASTER_SEED = 42
torch.manual_seed(MASTER_SEED)
random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
print(f'\nSeed = {MASTER_SEED}')


---
## 1 . Install dependencies

In [ ]:
EXTRA_PKGS = [
    'timm>=0.9.0',
    'opencv-python-headless>=4.8.0',
    'onnx>=1.14.0',
    'onnxruntime>=1.15.0',
    'grad-cam>=1.4.0',
    'pydicom>=2.4.0',
    'tqdm>=4.65.0',
    'wandb>=0.15.0',
]

import subprocess, sys
for pkg in EXTRA_PKGS:
    ret = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True,
    )
    print(f'{"OK" if ret.returncode == 0 else "FAIL"} {pkg}')
print('Done.')


---
## 2 . Clone source code from GitHub

Pulls directly from `tripathiji1312/cough-vision` -- no need to upload
the repo as a Kaggle dataset. Internet must be ON in notebook settings.


In [ ]:
import pathlib, sys, subprocess

GITHUB_REPO = 'https://github.com/tripathiji1312/cough-vision.git'
CLONE_DIR   = WORK_DIR / 'cough-vision'

if not CLONE_DIR.exists():
    print(f'Cloning {GITHUB_REPO} ...')
    ret = subprocess.run(
        ['git', 'clone', '--depth', '1', GITHUB_REPO, str(CLONE_DIR)],
        capture_output=True, text=True,
    )
    if ret.returncode != 0:
        print('Clone failed:', ret.stderr[-400:])
        raise RuntimeError('git clone failed -- check Internet is ON in settings')
    print('Cloned OK')
else:
    print(f'Already cloned at {CLONE_DIR}')

SRC_ROOT = CLONE_DIR / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
print(f'SRC_ROOT added to sys.path: {SRC_ROOT}')

# Quick smoke-test
from config import get_config
cfg_test = get_config('densenet_vit_ensemble')
print(f'Config OK -- CNN={cfg_test.cnn.name}, ViT={cfg_test.vit.name}')
del cfg_test


---
## 3 . Dataset paths

### Kaggle input layout expected
```
/kaggle/input/
  pulmonary-chest-xray-abnormalities/
    shenzhen/CXR_png/            <- 662 PNGs (336 TB, 326 normal)
    montgomery/CXR_png/          <- 138 PNGs (58 TB, 80 normal)
  tbx11k/
    imgs/train/  imgs/val/       <- 11200 PNGs
    labels.csv
  tuberculosis-tb-chest-xray-dataset/  (optional)
    TB_Chest_Radiography_Database/
  data/                          (optional NIH CXR14 negatives)
    images/
```


In [ ]:
import pathlib, csv, json, random, math
import numpy as np

KAGGLE_INPUT = pathlib.Path('/kaggle/input') if IS_KAGGLE else pathlib.Path('data')

# -- Primary TB datasets (required) -----------------------------------
SHENZHEN_IMG    = KAGGLE_INPUT / 'pulmonary-chest-xray-abnormalities' / 'shenzhen'   / 'CXR_png'
MONTGOMERY_IMG  = KAGGLE_INPUT / 'pulmonary-chest-xray-abnormalities' / 'montgomery' / 'CXR_png'
TBX11K_DIR      = KAGGLE_INPUT / 'tbx11k'

# -- Optional extra TB dataset ----------------------------------------
TAWSIF_DIR = KAGGLE_INPUT / 'tuberculosis-tb-chest-xray-dataset' / 'TB_Chest_Radiography_Database'

# -- Optional NIH CXR14 negatives (not pretraining) -------------------
NIH_IMG_DIR = KAGGLE_INPUT / 'data' / 'images'

DATA_DIR = WORK_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

print('Dataset availability:')
for name, path in [
    ('Shenzhen (required)',    SHENZHEN_IMG),
    ('Montgomery (required)',  MONTGOMERY_IMG),
    ('TBX11K (recommended)',   TBX11K_DIR),
    ('Tawsif TB (optional)',   TAWSIF_DIR),
    ('NIH CXR14 (optional)',   NIH_IMG_DIR),
]:
    exists = path.exists()
    print(f'  {"FOUND " if exists else "MISS  "} {name}: {path}')


In [ ]:
records = []
random.seed(42)

# -- Shenzhen (CHNCXR_XXXX_1.png = TB, _0.png = Normal) ---------------
if SHENZHEN_IMG.exists():
    for img in sorted(SHENZHEN_IMG.glob('*.png')):
        tb = int(img.stem.split('_')[-1])
        records.append({'image_path': str(img), 'tb_label': tb,
                         'findings_label': '0,0,0,0,0,0',
                         'active_inactive_label': -1,
                         'site': 'shenzhen', 'view_position': 'PA'})
    print(f'Shenzhen  : {len([r for r in records if r["site"]=="shenzhen"])} images')

# -- Montgomery (same naming convention) -------------------------------
if MONTGOMERY_IMG.exists():
    before = len(records)
    for img in sorted(MONTGOMERY_IMG.glob('*.png')):
        tb = int(img.stem.split('_')[-1])
        records.append({'image_path': str(img), 'tb_label': tb,
                         'findings_label': '0,0,0,0,0,0',
                         'active_inactive_label': -1,
                         'site': 'montgomery', 'view_position': 'PA'})
    print(f'Montgomery: {len(records)-before} images')

# -- TBX11K ------------------------------------------------------------
if TBX11K_DIR.exists():
    before = len(records)
    lbl_csv = TBX11K_DIR / 'labels.csv'
    if lbl_csv.exists():
        with open(lbl_csv) as f:
            for row in csv.DictReader(f):
                fname = row.get('file_name', row.get('image_path', ''))
                img_p = TBX11K_DIR / 'imgs' / fname
                if not img_p.exists():
                    continue
                lbl_raw = str(row.get('label', row.get('class', '0')))
                tb = 1 if ('tb' in lbl_raw.lower() and 'non' not in lbl_raw.lower()) else 0
                records.append({'image_path': str(img_p), 'tb_label': tb,
                                 'findings_label': '0,0,0,0,0,0',
                                 'active_inactive_label': -1,
                                 'site': 'tbx11k', 'view_position': 'PA'})
    else:
        # Try scanning image folders directly
        for split in ['train', 'val']:
            for img in (TBX11K_DIR / 'imgs' / split).glob('*.png'):
                records.append({'image_path': str(img), 'tb_label': -1,
                                 'findings_label': '0,0,0,0,0,0',
                                 'active_inactive_label': -1,
                                 'site': 'tbx11k', 'view_position': 'PA'})
    print(f'TBX11K    : {len(records)-before} images')

# -- Tawsif TB dataset (optional) --------------------------------------
if TAWSIF_DIR.exists():
    before = len(records)
    for sub, tb in [('Normal', 0), ('Tuberculosis', 1)]:
        d = TAWSIF_DIR / sub
        if d.exists():
            for img in d.glob('*.png'):
                records.append({'image_path': str(img), 'tb_label': tb,
                                 'findings_label': '0,0,0,0,0,0',
                                 'active_inactive_label': -1,
                                 'site': 'tawsif', 'view_position': 'PA'})
    print(f'Tawsif TB : {len(records)-before} images')

# -- NIH CXR14 negatives (optional, capped at 5000 to avoid imbalance) -
NIH_CAP = 5000
if NIH_IMG_DIR.exists():
    before   = len(records)
    nih_imgs = list(NIH_IMG_DIR.glob('*.png'))[:NIH_CAP]
    for img in nih_imgs:
        records.append({'image_path': str(img), 'tb_label': 0,
                         'findings_label': '0,0,0,0,0,0',
                         'active_inactive_label': -1,
                         'site': 'nih_cxr14', 'view_position': 'PA'})
    print(f'NIH CXR14 : {len(records)-before} images (capped at {NIH_CAP})')

print(f'\nTotal     : {len(records)}')
print(f'  TB pos  : {sum(r["tb_label"]==1 for r in records)}')
print(f'  Normal  : {sum(r["tb_label"]==0 for r in records)}')
print(f'  Unknown : {sum(r["tb_label"]==-1 for r in records)}')

if len(records) == 0:
    print('\nWARNING: No real datasets found -- running DEMO MODE')
    demo_dir = DATA_DIR / 'demo'
    demo_dir.mkdir(exist_ok=True)
    import cv2
    for i in range(200):
        arr = np.random.randint(20, 220, (256, 256), dtype=np.uint8)
        p   = demo_dir / f'img_{i:04d}.png'
        cv2.imwrite(str(p), arr)
        records.append({'image_path': str(p), 'tb_label': int(i % 5 == 0),
                         'findings_label': '0,0,0,0,0,0',
                         'active_inactive_label': -1,
                         'site': 'demo', 'view_position': 'PA'})
    print(f'Demo records: {len(records)}')


In [ ]:
from data.dataset import stratified_split

train_recs, val_recs, test_recs = stratified_split(
    records, val_fraction=0.10, test_fraction=0.10, seed=42
)

FIELDNAMES = ['image_path','tb_label','findings_label',
              'active_inactive_label','site','view_position','split']

def write_csv(path, rows, split_name):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=FIELDNAMES)
        w.writeheader()
        for r in rows:
            r['split'] = split_name
            w.writerow({k: r.get(k, '') for k in FIELDNAMES})

write_csv(DATA_DIR / 'train.csv', train_recs, 'train')
write_csv(DATA_DIR / 'val.csv',   val_recs,   'val')
write_csv(DATA_DIR / 'test.csv',  test_recs,  'test')

print(f'train : {len(train_recs):>5}  '
      f'(TB={sum(r["tb_label"]==1 for r in train_recs)}, '
      f'Normal={sum(r["tb_label"]==0 for r in train_recs)})')
print(f'val   : {len(val_recs):>5}  '
      f'(TB={sum(r["tb_label"]==1 for r in val_recs)}, '
      f'Normal={sum(r["tb_label"]==0 for r in val_recs)})')
print(f'test  : {len(test_recs):>5}  '
      f'(TB={sum(r["tb_label"]==1 for r in test_recs)}, '
      f'Normal={sum(r["tb_label"]==0 for r in test_recs)})')


---
## 4 . Preprocessing sanity check

In [ ]:
import matplotlib.pyplot as plt
import cv2
from data.preprocessing import load_image, apply_clahe, apply_gaussian_denoise, preprocess_cxr, passes_qc

sample = train_recs[0]
raw    = load_image(sample['image_path'])
clhe   = apply_clahe(raw, clip_limit=2.5)
dns    = apply_gaussian_denoise(clhe, sigma=0.5)
out    = preprocess_cxr(raw, target_size=224)

print(f'Image : {sample["image_path"]}')
print(f'Label : {"TB" if sample["tb_label"] else "Normal"}')
print(f'Raw   : {raw.shape}  QC={"PASS" if passes_qc(raw) else "FAIL"}')
print(f'Output: {out.shape}  {out.dtype}  min={out.min():.3f} max={out.max():.3f}')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes, [raw, clhe, dns, out[0]],
                           ['Raw CXR', 'CLAHE', 'Denoised', 'Net input ch0']):
    ax.imshow(img, cmap='gray', vmin=img.min(), vmax=img.max())
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle('Preprocessing pipeline', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'preprocessing_check.png', dpi=100)
plt.show()


---
## 5 . Hyperparameters

> `batch_size` is per GPU. Effective = batch_size x WORLD_SIZE x accum_steps


In [ ]:
HP = dict(
    preset          = 'densenet_vit_ensemble',
    cnn_backbone    = 'densenet121',
    vit_backbone    = 'vit_small_patch16_384',
    pretrained      = 'imagenet',
    max_epochs      = 60,
    freeze_epochs   = 3,
    batch_size      = 16,           # per GPU -> 16x2x2 = 64 effective
    accum_steps     = 2,
    backbone_lr     = 1e-5,
    head_lr         = 1e-3,
    weight_decay    = 1e-4,
    warmup_epochs   = 5,
    patience        = 10,
    focal_gamma     = 2.0,
    focal_alpha     = 0.75,
    label_smoothing = 0.1,
    cls_weight      = 1.0,
    findings_weight = 0.3,
    active_weight   = 0.2,
    use_cutmix      = True,
    use_mixup       = True,
    pos_weight      = 5.0,
    who_sensitivity = 0.90,
    mixed_precision = True,
    seed            = 42,
    num_workers     = 2,
    port            = 12355,
)

eff_bs = HP['batch_size'] * max(1, WORLD_SIZE) * HP['accum_steps']
print(f'Effective batch = {eff_bs}')
print(f'  {HP["batch_size"]} per-GPU x {max(1,WORLD_SIZE)} GPUs x {HP["accum_steps"]} accum')

with open(WORK_DIR / 'hyperparameters.json', 'w') as f:
    json.dump(HP, f, indent=2)
print('Saved -> hyperparameters.json')


---
## 6 . DDP utilities

In [ ]:
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Sampler
from torch.utils.data.distributed import DistributedSampler
import socket, math, os, datetime


def find_free_port(preferred: int = 12355) -> int:
    for port in [preferred] + list(range(49152, 49252)):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                s.bind(('', port))
                return port
        except OSError:
            continue
    raise RuntimeError('No free port found')


def setup_ddp(rank: int, world_size: int, port: int) -> None:
    os.environ.update({
        'MASTER_ADDR': 'localhost',
        'MASTER_PORT': str(port),
        'NCCL_P2P_DISABLE': '1',   # Kaggle T4 uses PCIe not NVLink
        'NCCL_IB_DISABLE':  '1',
    })
    dist.init_process_group(
        backend='nccl', rank=rank, world_size=world_size,
        timeout=datetime.timedelta(minutes=60),
    )
    torch.cuda.set_device(rank)


def cleanup_ddp() -> None:
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


class DistributedWeightedSampler(Sampler):
    """
    Combines WeightedRandomSampler + DistributedSampler.
    Each rank gets a unique, class-balanced shard of the dataset.
    """
    def __init__(self, weights: list, num_replicas: int, rank: int,
                 replacement: bool = True, seed: int = 42) -> None:
        self.weights      = torch.as_tensor(weights, dtype=torch.float64)
        self.num_replicas = num_replicas
        self.rank         = rank
        self.replacement  = replacement
        self.seed         = seed
        self.epoch        = 0
        self.num_samples  = math.ceil(len(weights) / num_replicas)
        self.total_size   = self.num_samples * num_replicas

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch * 10007)
        indices = torch.multinomial(
            self.weights, self.total_size,
            replacement=self.replacement, generator=g,
        ).tolist()
        return iter(indices[self.rank : self.total_size : self.num_replicas])

    def __len__(self) -> int:
        return self.num_samples


print('DDP utilities ready.')
print(f'  WORLD_SIZE = {WORLD_SIZE}')


---
## 7 . Model smoke-test

In [ ]:
from config import get_config
from models import build_ensemble

cfg = get_config(HP['preset'])
cfg.cnn.name         = HP['cnn_backbone']
cfg.vit.name         = HP['vit_backbone']
cfg.cnn.pretrained   = HP['pretrained']
cfg.vit.pretrained   = HP['pretrained']
cfg.train.output_dir = str(WORK_DIR / 'checkpoints')

_dev = torch.device('cuda:0' if WORLD_SIZE > 0 else 'cpu')
_m   = build_ensemble(cfg).to(_dev)

total_p = sum(p.numel() for p in _m.parameters())
print(f'Total params : {total_p/1e6:.2f} M')

_m.eval()
with torch.no_grad():
    _o = _m(torch.zeros(2, 3, 224, 224, device=_dev))

print(f'Output keys  : {list(_o.keys())}')
print(f'tb_prob      : {_o["tb_prob"].tolist()}')
print(f'findings     : {_o["findings_logits"].shape}')
print('Forward pass OK')

del _m, _o
if _dev.type == 'cuda':
    torch.cuda.empty_cache()


---
## 8 . DDP training worker

`ddp_worker(rank, world_size, args)` is the complete pipeline for one GPU.
Launched by `mp.spawn` for T4x2, or called directly for single GPU.


In [ ]:
def ddp_worker(rank: int, world_size: int, args: dict) -> None:
    """
    Complete DDP training worker -- one per GPU.
    All imports are inside because mp.spawn creates fresh processes.
    """
    import os, sys, math, time, json, datetime, warnings, pathlib

    sys.path.insert(0, args['src_root'])
    setup_ddp(rank, world_size, args['port'])
    is_main = (rank == 0)
    device  = torch.device(f'cuda:{rank}')
    torch.manual_seed(args['seed'] + rank * 137)

    if is_main:
        eff = args['batch_size'] * world_size * args['accum_steps']
        print(f'[DDP] {world_size} processes | batch/GPU={args["batch_size"]} | effective={eff}')

    # -- imports --------------------------------------------------------
    from config import get_config
    from models import build_ensemble
    from training.losses import MultiTaskLoss
    from training.scheduler import build_optimizer, cosine_schedule_with_warmup
    from data.dataset import TBDataset, compute_sample_weights
    from data.augmentation import get_train_transform, get_inference_transform, CutMixMixUpCollator
    from torch.utils.data import DataLoader
    from torch.utils.data.distributed import DistributedSampler
    import numpy as np

    _roc_auc = None
    try:
        from sklearn.metrics import roc_auc_score as _roc_auc
    except ImportError:
        pass

    # -- config ---------------------------------------------------------
    cfg = get_config(args['preset'])
    cfg.cnn.name       = args['cnn_backbone']
    cfg.vit.name       = args['vit_backbone']
    cfg.cnn.pretrained = args['pretrained']
    cfg.vit.pretrained = args['pretrained']
    output_dir = pathlib.Path(args['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)

    # -- model -> DDP ---------------------------------------------------
    model = build_ensemble(cfg).to(device)
    model = DDP(model, device_ids=[rank], output_device=rank,
                find_unused_parameters=False)

    # -- datasets -------------------------------------------------------
    root = pathlib.Path('/')
    train_ds = TBDataset(
        csv_path=args['train_csv'], image_root=root, split='train',
        transform=get_train_transform(
            image_size=224, rotation_degrees=10.0,
            translate_fraction=0.05, scale_range=(0.85, 1.15),
            brightness_jitter=0.20, contrast_jitter=0.20,
        ),
    )
    val_ds = TBDataset(
        csv_path=args['val_csv'], image_root=root, split='val',
        transform=get_inference_transform(image_size=224),
    )

    # -- samplers -------------------------------------------------------
    tb_lbls = [int(r.get('tb_label', 0)) for r in train_ds.records]
    weights = compute_sample_weights(tb_lbls, pos_weight=args['pos_weight'])
    train_sampler = DistributedWeightedSampler(
        weights, num_replicas=world_size, rank=rank, seed=args['seed'],
    )
    val_sampler = DistributedSampler(
        val_ds, num_replicas=world_size, rank=rank, shuffle=False,
    )

    collator = CutMixMixUpCollator(
        n_classes=2, cutmix_alpha=1.0, mixup_alpha=0.4,
        cutmix_prob=0.5 if args['use_cutmix'] else 0.0,
        mixup_prob=0.5  if args['use_mixup']  else 0.0,
    )
    nw = args['num_workers']
    train_loader = DataLoader(
        train_ds, sampler=train_sampler, batch_size=args['batch_size'],
        num_workers=nw, pin_memory=True, drop_last=True,
        collate_fn=collator, persistent_workers=(nw > 0),
    )
    val_loader = DataLoader(
        val_ds, sampler=val_sampler, batch_size=args['batch_size'] * 2,
        num_workers=nw, pin_memory=True, persistent_workers=(nw > 0),
    )

    # -- loss / optim / scaler ------------------------------------------
    criterion = MultiTaskLoss(
        cls_weight=args['cls_weight'], findings_weight=args['findings_weight'],
        active_weight=args['active_weight'], focal_gamma=args['focal_gamma'],
        focal_alpha=args['focal_alpha'], label_smoothing=args['label_smoothing'],
    ).to(device)

    accum   = args['accum_steps']
    max_ep  = args['max_epochs']
    frz_ep  = args['freeze_epochs']
    wrm_ep  = args['warmup_epochs']
    pat     = args['patience']
    mp_flag = args['mixed_precision']

    model.freeze_backbones()
    optimizer = build_optimizer(model, args['backbone_lr'], args['head_lr'], args['weight_decay'])
    steps_ep     = len(train_loader)
    total_steps  = max_ep * steps_ep
    warmup_steps = wrm_ep * steps_ep
    scheduler    = cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = torch.cuda.amp.GradScaler(enabled=mp_flag)

    best_auc = 0.0
    no_improve = 0
    history: list = []

    # -- training loop --------------------------------------------------
    for epoch in range(max_ep):
        t0 = time.time()
        train_sampler.set_epoch(epoch)

        if epoch == frz_ep:
            model.unfreeze_backbones()
            optimizer = build_optimizer(model, args['backbone_lr'], args['head_lr'], args['weight_decay'])
            scheduler = cosine_schedule_with_warmup(optimizer, 0, total_steps)
            if is_main:
                print(f'[Epoch {epoch:03d}] Phase 2b: backbones unfrozen')

        # train
        model.train()
        run_loss = torch.zeros(1, device=device)
        n_steps  = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            if isinstance(batch[1], dict):
                images    = batch[0].to(device, non_blocking=True)
                ld        = batch[1]
                tb_labels = ld['labels_a'].to(device, non_blocking=True)
                lam       = float(ld.get('lam', 1.0))
                labels_b  = ld['labels_b'].to(device, non_blocking=True)
                findings  = torch.zeros(images.shape[0], 6, device=device)
                active    = torch.full((images.shape[0],), -1, dtype=torch.long, device=device)
            else:
                images, tb_raw, find_raw, act_raw, *_ = batch
                images    = images.to(device, non_blocking=True)
                tb_labels = tb_raw.to(device, non_blocking=True)
                findings  = find_raw.to(device, non_blocking=True)
                active    = act_raw.to(device, non_blocking=True)
                lam, labels_b = 1.0, None

            with torch.cuda.amp.autocast(enabled=mp_flag):
                out = model(images)
                ld_ = criterion(out, tb_labels, findings, active, lam=lam, labels_b=labels_b)
                loss = ld_['loss'] / accum

            scaler.scale(loss).backward()
            run_loss += loss.detach() * accum

            if (step + 1) % accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            n_steps += 1

        if (len(train_loader) % accum) != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step(); optimizer.zero_grad()

        dist.all_reduce(run_loss, op=dist.ReduceOp.AVG)
        avg_train = float(run_loss / max(1, n_steps))

        # validate
        model.eval()
        vl_t = torch.zeros(1, device=device)
        l_prob: list = []
        l_lbl:  list = []

        with torch.no_grad():
            for batch in val_loader:
                if isinstance(batch[1], dict):
                    images    = batch[0].to(device, non_blocking=True)
                    tb_labels = batch[1]['labels_a'].to(device, non_blocking=True)
                    findings  = torch.zeros(images.shape[0], 6, device=device)
                    active    = torch.full((images.shape[0],), -1, dtype=torch.long, device=device)
                else:
                    images, tb_raw, find_raw, act_raw, *_ = batch
                    images    = images.to(device, non_blocking=True)
                    tb_labels = tb_raw.to(device, non_blocking=True)
                    findings  = find_raw.to(device, non_blocking=True)
                    active    = act_raw.to(device, non_blocking=True)

                out = model(images)
                vl_t += criterion(out, tb_labels, findings, active)['loss'].detach()
                l_prob.extend(out['tb_prob'].cpu().tolist())
                lbs = (tb_labels.argmax(dim=-1) if tb_labels.dim() > 1 else tb_labels).cpu().tolist()
                l_lbl.extend(lbs)

        dist.all_reduce(vl_t, op=dist.ReduceOp.AVG)
        avg_val = float(vl_t / max(1, len(val_loader)))

        # gather predictions from all ranks for global AUC
        all_p = [None] * world_size
        all_l = [None] * world_size
        dist.all_gather_object(all_p, l_prob)
        dist.all_gather_object(all_l, l_lbl)

        if is_main:
            flat_p = [v for sub in all_p for v in sub]
            flat_l = [v for sub in all_l for v in sub]
            val_auc = 0.5
            if _roc_auc is not None and len(set(flat_l)) > 1:
                try:
                    val_auc = float(_roc_auc(np.array(flat_l), np.array(flat_p)))
                except Exception:
                    pass

            elapsed = time.time() - t0
            row = dict(epoch=epoch, train_loss=avg_train, val_loss=avg_val,
                       val_auc=val_auc, elapsed_s=elapsed)
            history.append(row)
            print(f'Epoch {epoch:03d} | train={avg_train:.4f} | val_loss={avg_val:.4f} | '
                  f'val_auc={val_auc:.4f} | {elapsed:.1f}s')

            if val_auc > best_auc:
                best_auc = val_auc; no_improve = 0
                try:
                    torch.save({'epoch': epoch,
                                'model_state_dict': model.module.state_dict(),
                                'optimizer_state_dict': optimizer.state_dict(),
                                'val_auc': val_auc, 'history': history},
                               output_dir / 'best_model.pt')
                except Exception as e:
                    warnings.warn(f'Checkpoint save failed: {e}')
            else:
                no_improve += 1

            try:
                torch.save({'epoch': epoch, 'model_state_dict': model.module.state_dict()},
                           output_dir / 'last_model.pt')
            except Exception:
                pass
            should_stop = int(no_improve >= pat)
        else:
            should_stop = 0

        stop_t = torch.tensor(should_stop, dtype=torch.int32, device=device)
        dist.broadcast(stop_t, src=0)
        dist.barrier()

        if stop_t.item():
            if is_main:
                print(f'Early stopping at epoch {epoch} (patience={pat} exhausted)')
            break

    if is_main:
        print(f'\nDone. Best val AUC = {best_auc:.4f}')
        (output_dir / 'history.json').write_text(json.dumps(history, indent=2))

    cleanup_ddp()


print('ddp_worker defined.')


---
## 9 . Launch training

- **T4x2**: `mp.spawn` starts 2 worker processes in parallel.
- **Single GPU / CPU**: worker called directly.


In [ ]:
# Optional W&B
wandb_run = None
try:
    import wandb
    key = os.environ.get('WANDB_API_KEY', '')
    if key:
        wandb.login(key=key, relogin=True)
        wandb_run = wandb.init(
            project='cough-vision',
            name=f'{HP["preset"]}_{HP["max_epochs"]}ep_{WORLD_SIZE}gpu',
            config=HP, dir=str(WORK_DIR),
        )
        print('W&B:', wandb_run.url)
    else:
        print('W&B skipped -- add WANDB_API_KEY in Kaggle Secrets to enable')
except ImportError:
    print('wandb not installed')

CKPT_DIR = str(WORK_DIR / 'checkpoints')

ARGS = dict(
    src_root        = str(SRC_ROOT),
    preset          = HP['preset'],
    cnn_backbone    = HP['cnn_backbone'],
    vit_backbone    = HP['vit_backbone'],
    pretrained      = HP['pretrained'],
    max_epochs      = HP['max_epochs'],
    freeze_epochs   = HP['freeze_epochs'],
    batch_size      = HP['batch_size'],
    accum_steps     = HP['accum_steps'],
    backbone_lr     = HP['backbone_lr'],
    head_lr         = HP['head_lr'],
    weight_decay    = HP['weight_decay'],
    warmup_epochs   = HP['warmup_epochs'],
    patience        = HP['patience'],
    focal_gamma     = HP['focal_gamma'],
    focal_alpha     = HP['focal_alpha'],
    label_smoothing = HP['label_smoothing'],
    cls_weight      = HP['cls_weight'],
    findings_weight = HP['findings_weight'],
    active_weight   = HP['active_weight'],
    use_cutmix      = HP['use_cutmix'],
    use_mixup       = HP['use_mixup'],
    pos_weight      = HP['pos_weight'],
    mixed_precision = HP['mixed_precision'],
    seed            = HP['seed'],
    num_workers     = HP['num_workers'],
    train_csv       = str(DATA_DIR / 'train.csv'),
    val_csv         = str(DATA_DIR / 'val.csv'),
    output_dir      = CKPT_DIR,
    port            = find_free_port(HP['port']),
)

print(f'MASTER_PORT : {ARGS["port"]}')
print(f'WORLD_SIZE  : {WORLD_SIZE}')
print(f'Strategy    : {"DDP mp.spawn (T4x2)" if WORLD_SIZE > 1 else "Single process"}')
print()

if WORLD_SIZE > 1:
    mp.spawn(ddp_worker, args=(WORLD_SIZE, ARGS), nprocs=WORLD_SIZE, join=True)
else:
    ddp_worker(rank=0, world_size=1, args=ARGS)

if wandb_run is not None:
    try:
        wandb_run.finish()
    except Exception:
        pass

print('Training complete.')


---
## 10 . Training curves

In [ ]:
import matplotlib.pyplot as plt, json, pathlib

hist_path = pathlib.Path(CKPT_DIR) / 'history.json'
if not hist_path.exists():
    bp = pathlib.Path(CKPT_DIR) / 'best_model.pt'
    history = torch.load(str(bp), map_location='cpu').get('history', []) if bp.exists() else []
else:
    history = json.loads(hist_path.read_text())

if not history:
    print('No history yet -- run training first.')
else:
    epochs     = [h['epoch']      for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss   = [h['val_loss']   for h in history]
    val_auc    = [h['val_auc']    for h in history]
    best_ep  = max(history, key=lambda h: h['val_auc'])['epoch']
    best_auc = max(h['val_auc'] for h in history)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, train_loss, label='Train loss', color='steelblue', lw=2)
    ax1.plot(epochs, val_loss,   label='Val loss',   color='tomato',    lw=2)
    ax1.axvline(HP['freeze_epochs'], color='gray', ls='--', alpha=0.6)
    ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss curves'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, val_auc, color='forestgreen', marker='o', ms=3, lw=2)
    ax2.axhline(0.90, color='red', ls='--', lw=1.5, label='WHO 90% target')
    ax2.scatter([best_ep], [best_auc], color='gold', s=120, zorder=5,
                label=f'Best AUC={best_auc:.4f} (ep {best_ep})')
    ax2.set(xlabel='Epoch', ylabel='AUC-ROC', title='Validation AUC (all GPUs)')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.suptitle('cough-vision Training Curves', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'training_curves.png', dpi=120)
    plt.show()
    print(f'Best val AUC = {best_auc:.4f} at epoch {best_ep}')


---
## 11 . Evaluation -- WHO TPP report

In [ ]:
import json, numpy as np
from evaluation.metrics import evaluate, find_who_threshold, auc_metrics
from models import build_ensemble
from data.dataset import TBDataset
from data.augmentation import get_inference_transform
from torch.utils.data import DataLoader

ckpt_path = pathlib.Path(CKPT_DIR) / 'best_model.pt'
eval_model = build_ensemble(get_config(HP['preset']))
eval_model.cfg_cnn_name = HP['cnn_backbone']

if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location='cpu')
    eval_model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model (epoch={ckpt.get("epoch","?")} val_auc={ckpt.get("val_auc",0):.4f})')
else:
    print('WARNING: no checkpoint -- evaluating untrained model')

eval_dev = torch.device('cuda:0' if WORLD_SIZE > 0 else 'cpu')
eval_model = eval_model.to(eval_dev).eval()

test_ds = TBDataset(csv_path=DATA_DIR/'test.csv', image_root=pathlib.Path('/'),
                    split='test', transform=get_inference_transform(224))
test_loader = DataLoader(test_ds, batch_size=HP['batch_size']*2,
                         shuffle=False, num_workers=HP['num_workers'],
                         pin_memory=(eval_dev.type=='cuda'))

all_probs, all_lbls = [], []
with torch.no_grad():
    for batch in test_loader:
        imgs, lbls, *_ = batch
        out = eval_model(imgs.to(eval_dev))
        all_probs.extend(out['tb_prob'].cpu().tolist())
        all_lbls.extend(lbls.cpu().tolist())

y_true = np.array(all_lbls,  dtype=int)
y_prob = np.array(all_probs, dtype=float)

report = evaluate(y_true, y_prob,
                  sensitivity_target=HP['who_sensitivity'],
                  n_bootstrap=1000, site='test_set')
(WORK_DIR / 'eval_report.json').write_text(json.dumps(report, indent=2, default=str))

print('=' * 55 + '\nEVALUATION\n' + '=' * 55)
try:
    print(f'AUC-ROC : {report["auc"]["auc_roc"]:.4f}  '
          f'CI [{report["auc_roc_ci"]["ci_lower"]:.4f}, {report["auc_roc_ci"]["ci_upper"]:.4f}]')
    print(f'AUC-PR  : {report["auc"]["auc_pr"]:.4f}')
except (KeyError, TypeError): pass
try:
    who = report['who_operating_point']
    mets = report['metrics_at_who_threshold']
    print(f'WHO TPP : {"MET" if who.get("who_tpp_met") else "NOT MET"}')
    print(f'  Threshold   : {who["threshold"]:.4f}')
    print(f'  Sensitivity : {who["sensitivity"]:.1%}')
    print(f'  Specificity : {who["specificity"]:.1%}')
    print(f'  PPV / NPV   : {mets.get("ppv",0):.1%} / {mets.get("npv",0):.1%}')
    print(f'  F1 / MCC    : {mets.get("f1",0):.4f} / {mets.get("mcc",0):.4f}')
except (KeyError, TypeError): pass
print('=' * 55)


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fpr, tpr, _ = roc_curve(y_true, y_prob)
prec, rec, _ = precision_recall_curve(y_true, y_prob)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.plot(fpr, tpr, lw=2, color='steelblue', label=f'ROC AUC = {auc(fpr,tpr):.4f}')
ax1.plot([0,1],[0,1],'k--',alpha=0.4)
ax1.axhline(0.90, color='red', ls=':', alpha=0.6, label='90% sensitivity')
ax1.set(xlabel='FPR', ylabel='TPR', title='ROC Curve'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(rec, prec, lw=2, color='forestgreen', label=f'PR AUC = {auc(rec,prec):.4f}')
ax2.axhline(y_true.mean(), color='gray', ls='--', label=f'Prevalence={y_true.mean():.1%}')
ax2.set(xlabel='Recall', ylabel='Precision', title='PR Curve'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('cough-vision Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'roc_pr_curves.png', dpi=120)
plt.show()


---
## 12 . Grad-CAM

In [ ]:
import cv2
from data.preprocessing import load_image, apply_clahe
from data.augmentation import get_inference_transform
import PIL.Image as PilImage

eval_model.enable_gradcam()

tb_cases = [r for r in test_ds.records if int(r.get('tb_label',0))==1][:4]
if not tb_cases: tb_cases = test_ds.records[:4]

_tf = get_inference_transform(224)
fig, axes = plt.subplots(len(tb_cases), 3, figsize=(13, 4*len(tb_cases)))
if len(tb_cases) == 1: axes = [axes]

for i, rec in enumerate(tb_cases):
    raw  = load_image(rec['image_path'])
    H, W = raw.shape
    pil  = PilImage.fromarray(apply_clahe(raw)).convert('RGB')
    img_t = _tf(pil).unsqueeze(0).to(eval_dev)

    eval_model.eval()
    with torch.no_grad():
        out = eval_model(img_t)
    score = float(out['tb_score'].cpu())

    try:
        hmap    = eval_model._gradcam(img_t, class_idx=1)
        overlay = eval_model._gradcam.overlay(hmap, raw)
    except Exception as e:
        print(f'CAM error: {e}')
        hmap    = np.zeros((H, W))
        overlay = cv2.cvtColor(raw, cv2.COLOR_GRAY2RGB)

    axes[i][0].imshow(raw, cmap='gray')
    axes[i][0].set_title(f'Original label={rec["tb_label"]}', fontsize=10)
    axes[i][0].axis('off')

    im = axes[i][1].imshow(cv2.resize(hmap,(W,H)), cmap='jet', vmin=0, vmax=1)
    axes[i][1].set_title(f'Grad-CAM score={score:.1f}/100', fontsize=10)
    axes[i][1].axis('off')
    plt.colorbar(im, ax=axes[i][1], fraction=0.046)

    axes[i][2].imshow(overlay)
    axes[i][2].set_title('Overlay', fontsize=10)
    axes[i][2].axis('off')

plt.suptitle('Grad-CAM TB cases', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'gradcam_tb_cases.png', dpi=120)
plt.show()
eval_model.disable_gradcam()


---
## 13 . Per-site calibration

In [ ]:
from evaluation.calibration import build_calibrator
from evaluation.metrics import find_who_threshold

cal = build_calibrator('platt')
cal.fit(y_prob, y_true, sensitivity_target=HP['who_sensitivity'])

cal_probs = cal.predict_proba(y_prob)
who_cal   = find_who_threshold(y_true, cal_probs, sensitivity_target=HP['who_sensitivity'])

print(f'Calibrated threshold : {cal.threshold_:.4f}')
print(f'Post-cal sensitivity : {who_cal["sensitivity"]:.1%}')
print(f'Post-cal specificity : {who_cal["specificity"]:.1%}')
print(f'WHO TPP met          : {"YES" if who_cal["who_tpp_met"] else "NO"}')

cal.save(WORK_DIR / 'calibrator.json')
print(f'Saved -> calibrator.json')


---
## 14 . ONNX export

In [ ]:
from deployment.export import export_onnx, optimise_onnx

try:
    exported = export_onnx(
        model=eval_model, output_path=WORK_DIR/'cough_vision.onnx',
        cnn_input_size=224, opset=17, dynamic_axes=True, model_version='v1.0.0',
    )
    print(f'Exported ({exported.stat().st_size/1e6:.1f} MB) -> {exported}')
    try:
        optimise_onnx(exported)
    except ImportError:
        print('onnxoptimizer not installed -- skip')
except Exception as e:
    print(f'ONNX export failed: {e}')


---
## 15 . Output artefacts

In [ ]:
import pathlib
artefacts = [
    ('best_model.pt',        pathlib.Path(CKPT_DIR) / 'best_model.pt'),
    ('calibrator.json',      WORK_DIR / 'calibrator.json'),
    ('eval_report.json',     WORK_DIR / 'eval_report.json'),
    ('hyperparameters.json', WORK_DIR / 'hyperparameters.json'),
    ('training_curves.png',  WORK_DIR / 'training_curves.png'),
    ('roc_pr_curves.png',    WORK_DIR / 'roc_pr_curves.png'),
    ('gradcam_tb_cases.png', WORK_DIR / 'gradcam_tb_cases.png'),
    ('cough_vision.onnx',    WORK_DIR / 'cough_vision.onnx'),
]
for name, path in artefacts:
    if path.exists():
        print(f'  SAVED  {name:<28} {path.stat().st_size/1e3:>8.1f} KB')
    else:
        print(f'  MISS   {name}')
print(f'\nAll output: {WORK_DIR}')


---
## 16 . Next steps

| Step | Action |
|---|---|
| External validation | Re-run eval on Montgomery with Shenzhen-trained model |
| Better init | Download MoCo-CXR checkpoint; set `pretrained='mocov3_cxr'` |
| Edge export | Use `edge_efficientnet` preset + quantize |
| Calibrate | `scripts/deploy.py calibrate` on 50-200 local CXRs |
